# Explicabilité ORACLE — prédictions + DAG

**But** : produire un maximum de graphes d'explicabilité + prédictions pour le modèle **ORACLE**, vite.

3 niveaux (indépendants — si un échoue les autres tournent) :
- **Niveau A — DAG structurel** : ne charge aucun modèle, lit `A_dag` + `G_phys`. Secondes.
- **Niveau B — Prédictions (caches)** : cartes cible/préd/erreur, extrêmes, biais. Pas de sampling.
- **Niveau C — (optionnel) sampling + ablation A_dag** : `RUN_TIER_C=False` par défaut.

Sorties dans un **dossier copiable** (Google Drive en Colab) + un **.zip** en fin de notebook.

> ⚠️ Cadrage honnête : le DAG est **imposé / verrouillé sur le prior physique**, pas découvert. Ces graphes montrent une structure **fidèle et auditable**, pas un mécanisme à fort levier.


In [ ]:
# === Niveau 0 : setup & chemins (bootstrap eprouve, repris du notebook d'entrainement) ===
import os, sys, json, math, subprocess, shlex
from pathlib import Path

GIT_URL    = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH = "four-node-causal"
LOCAL_PROJECT = "/content/climate_data"
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

def _run(cmd, check=False, timeout=None):
    print("$",cmd); rc=subprocess.call(shlex.split(cmd),timeout=timeout); print("  rc=",rc)
    if check and rc!=0: raise RuntimeError(cmd)
    return rc

if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"): drive.mount("/content/drive")
    _cands = [Path(LOCAL_PROJECT), Path("/content/drive/MyDrive/climate_data")]
    REPO_DIR = next((str(p) for p in _cands if (p/"src"/"st_cdgm").is_dir() or (p/".git").exists()), None)
    if REPO_DIR is None:
        Path(LOCAL_PROJECT).parent.mkdir(parents=True, exist_ok=True)
        _run(f"git clone --depth 1 -b {GIT_BRANCH} {GIT_URL} {LOCAL_PROJECT}", check=True)
        _run(f"git -C {LOCAL_PROJECT} pull --depth 200 origin {GIT_BRANCH}", timeout=180)
        REPO_DIR = LOCAL_PROJECT
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(Path(REPO_DIR)/"src"))
    try:
        import st_cdgm, omegaconf  # noqa
        print("[deps] imports critiques OK — pip saute")
    except Exception:
        _run(f"{shlex.quote(sys.executable)} -m pip install -q omegaconf diffusers==0.36.0 "
             f"transformers==4.57.6 torch-geometric networkx seaborn", timeout=600)
        _run(f"{shlex.quote(sys.executable)} -m pip install -q --no-deps -e {REPO_DIR}", timeout=120)
    DRIVE_ROOT = Path("/content/drive/MyDrive/climate_data")
else:
    _cur = Path(os.getcwd()).resolve()
    _cands = [_cur, *_cur.parents]
    try: _cands = [Path(__file__).resolve().parent, *Path(__file__).resolve().parents] + _cands
    except NameError: pass
    REPO_DIR = next((str(p) for p in _cands if (p/"src"/"st_cdgm").is_dir() or (p/"config").is_dir()), str(_cur))
    os.chdir(REPO_DIR)
    DRIVE_ROOT = Path(os.environ.get("CLIMATE_DRIVE_ROOT", REPO_DIR))

for _p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if _p not in sys.path: sys.path.insert(0, _p)
print("REPO_DIR =", REPO_DIR, "| Colab =", IN_COLAB)

# --- Chemins du modele ORACLE (11-node / oracle_v6_prime ; cles/chemins disque inchanges) ---
CKPT      = DRIVE_ROOT / "oracle_v6_prime/seed_42/v6_prime_seed42.pth"
CKPT_S1   = DRIVE_ROOT / "oracle_v6_prime/seed_42/stage1_seed42.pth"
CACHE     = DRIVE_ROOT / "oracle_v6_prime/seed_42/bs32b_cache_seed42.pt"

# Dossier de sortie COPIABLE (Drive en Colab -> persiste)
FIG_DIR = (DRIVE_ROOT/"explainability_report") if IN_COLAB else Path("results/explainability_report")
FIG_DIR.mkdir(parents=True, exist_ok=True)
import re, unicodedata
def slug(s):
    s=unicodedata.normalize("NFKD",str(s)).encode("ascii","ignore").decode()
    return re.sub(r"[^A-Za-z0-9]+","_",s).strip("_")
NAME = "ORACLE"
print("Figures (copiables) ->", FIG_DIR.resolve())

import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_context("notebook"); HAS_SNS=True
except Exception: HAS_SNS=False
try:
    import networkx as nx; HAS_NX=True
except Exception: HAS_NX=False
try:
    import torch; HAS_TORCH=True
except Exception: HAS_TORCH=False
plt.rcParams["figure.dpi"]=110
print("seaborn:",HAS_SNS,"| networkx:",HAS_NX,"| torch:",HAS_TORCH)

MAX_SAMPLES = 128
def _tload(p):
    import torch
    try:    return torch.load(p, map_location="cpu", weights_only=False, mmap=True)
    except Exception:
        return torch.load(p, map_location="cpu", weights_only=False)


In [ ]:
# === Niveau 0 : labels des nœuds + G_phys (prior physique) ===
try:
    from st_cdgm.training.physics_prior import VAR_LABELS_V6, EXPECTED_EDGES_V6, build_physical_mask
    LABELS = list(VAR_LABELS_V6)
    G = build_physical_mask(num_vars=len(LABELS), var_labels=LABELS, expected_edges=EXPECTED_EDGES_V6)
    G = np.asarray(G.detach().cpu().numpy() if hasattr(G,"detach") else G, dtype=float)
    print("[ok] G_phys importe depuis physics_prior")
except Exception as e:
    print("[fallback] physics_prior indisponible:", e)
    LABELS = ["GP850_spat","GP850->GP500","GP500_spat","GP500->GP250","GP250_spat","Q850","W500","IVT","U850","V850","SP_HR"]
    E = [(4,2,1),(2,0,1),(0,10,1),(1,2,1),(3,4,1),(0,5,1),(2,6,1),(5,7,1),(6,10,1),(7,10,1),
         (8,7,1),(9,7,-1),(8,10,1),(9,10,-1)]
    G=np.zeros((len(LABELS),len(LABELS)))
    for s,t,sg in E: G[s,t]=sg
print("ORACLE labels :", LABELS)
print("G_phys edges  :", int((G!=0).sum()))


In [ ]:
# === Niveau 0 : charger A_dag du modele ORACLE ===
def mask_diag(A):
    A=np.array(A,dtype=float); np.fill_diagonal(A,0.0); return A

def load_Adag():
    for p in (CKPT, CKPT_S1):
        if HAS_TORCH and Path(p).exists():
            ck=_tload(p)
            if ck.get("A_dag_final") is not None:
                print(f"[ORACLE] A_dag_final <- {Path(p).name}"); return mask_diag(np.array(ck["A_dag_final"]))
            sd=ck.get("rcn_cell_state_dict",{})
            if "A_dag" in sd:
                print(f"[ORACLE] A_dag <- {Path(p).name} state_dict"); return mask_diag(sd["A_dag"].cpu().numpy())
    print("[ORACLE] A_dag INTROUVABLE (checkpoint Drive requis — Colab)"); return None

A = load_Adag()
print("A shape:", None if A is None else A.shape)


In [ ]:
# === Niveau 0 : metriques + classification d'aretes ===
try:
    from path_c_plus.scripts.option_c_helpers import (
        compute_q_phys_binary, compute_q_phys_continuous, compute_skeleton_f1)
    HAS_QHELP=True
except Exception as e:
    HAS_QHELP=False; print("[warn] option_c_helpers indispo:", e)

def q_metrics(A, G):
    A=mask_diag(A); G=mask_diag(G); out={}
    if HAS_QHELP:
        try: qb,_,_,nx_=compute_q_phys_binary(A,G); out["q_phys_binary"]=float(qb); out["n_extra"]=int(nx_)
        except Exception: pass
        try: qc,coll=compute_q_phys_continuous(A,G); out["q_phys_cont"]=float(qc); out["collapsed"]=bool(coll)
        except Exception: pass
        try:
            _f1=compute_skeleton_f1(A,G); _f1=_f1[0] if isinstance(_f1,(tuple,list)) else _f1
            out["skeleton_f1"]=float(_f1)
        except Exception: pass
    thr=0.05; Ab=(np.abs(A)>thr); Gb=(np.abs(G)>0)
    tp=int((Ab&Gb).sum()); fp=int((Ab&~Gb).sum()); fn=int((~Ab&Gb).sum())
    out.setdefault("skeleton_f1",(2*tp/(2*tp+fp+fn)) if (2*tp+fp+fn)>0 else 0.0)
    out.setdefault("n_extra",fp)
    out["sign_correct"]=int(((np.sign(A)==np.sign(G))&Gb).sum()); out["n_phys"]=int(Gb.sum())
    return out

def edge_classes(A,G,thr=0.05):
    A=mask_diag(A);G=mask_diag(G);q=A.shape[0];C=np.zeros((q,q));Ab=np.abs(A)>thr;Gb=np.abs(G)>0
    for i in range(q):
        for j in range(q):
            if Gb[i,j] and Ab[i,j]: C[i,j]=1.0 if np.sign(A[i,j])==np.sign(G[i,j]) else -1.0
            elif Ab[i,j] and not Gb[i,j]: C[i,j]=0.5
            elif Gb[i,j] and not Ab[i,j]: C[i,j]=-0.5
    return C

M = q_metrics(A,G) if A is not None else None
print("ORACLE metrics :", M)


In [ ]:
# === Niveau A — G1 : heatmaps A_dag appris + G_phys ===
def heat(ax,M,labels,title,vsym=True,cmap="RdBu_r"):
    M=mask_diag(M); v=np.abs(M).max() or 1.0
    im=ax.imshow(M,cmap=cmap,vmin=-v if vsym else 0,vmax=v)
    ax.set_xticks(range(len(labels)));ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels,rotation=90,fontsize=7);ax.set_yticklabels(labels,fontsize=7)
    ax.set_title(title,fontsize=10);plt.colorbar(im,ax=ax,fraction=0.046)
if A is not None:
    fig,axes=plt.subplots(1,2,figsize=(11,4.6))
    heat(axes[0],A,LABELS,f"A_dag appris — {NAME}")
    heat(axes[1],G,LABELS,f"G_phys (prior physique) — {NAME}")
    fig.suptitle("G1 — Adjacence causale : apprise vs physique",y=1.03,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/"G1_heatmaps_Adag.png",bbox_inches="tight");plt.show()
else: print("[G1 saute] A_dag absent (Colab requis)")


In [ ]:
# === Niveau A — G2 : confusion appris vs physique ===
from matplotlib.colors import to_rgb
_COL={"none":"#f7f7f7","correct":"#2ecc71","wrong":"#c0392b","extra":"#e67e22","missing":"#f6c1c1"}
def plot_conf(ax,A,G,labels,title):
    C=edge_classes(A,G);q=C.shape[0];rgb=np.ones((q,q,3))
    lut={1.0:"correct",-1.0:"wrong",0.5:"extra",-0.5:"missing"}
    for i in range(q):
        for j in range(q):
            if C[i,j]!=0: rgb[i,j]=to_rgb(_COL[lut[C[i,j]]])
    ax.imshow(rgb,interpolation="nearest")
    ax.set_xticks(range(len(labels)));ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels,rotation=90,fontsize=7);ax.set_yticklabels(labels,fontsize=7)
    ax.set_title(title,fontsize=10)
if A is not None:
    fig,ax=plt.subplots(figsize=(6,5.4))
    plot_conf(ax,A,G,LABELS,f"{NAME} — skelF1={M.get('skeleton_f1',0):.2f}  n_extra={M.get('n_extra','?')}  signes={M.get('sign_correct','?')}/{M.get('n_phys','?')}")
    import matplotlib.patches as mp
    fig.legend(handles=[mp.Patch(color="#2ecc71",label="correct"),mp.Patch(color="#c0392b",label="signe inverse"),
               mp.Patch(color="#f6c1c1",label="manquant"),mp.Patch(color="#e67e22",label="extra")],
               loc="lower center",ncol=4,fontsize=9,bbox_to_anchor=(0.5,-0.08))
    fig.suptitle("G2 — Récupération structurelle (confusion)",y=1.02,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/"G2_confusion.png",bbox_inches="tight");plt.show()
else: print("[G2 saute]")


In [ ]:
# === Niveau A — G3 : DAG en graphe de réseau ===
POS={0:(0,0),1:(1,0.6),2:(0,1),3:(1,1.6),4:(0,2),5:(1.8,0.3),6:(1.8,1.3),7:(2.6,0.8),
     8:(3.2,0.0),9:(3.2,1.6),10:(1.4,-1.2)}
def draw_dag(ax,A,G,labels,pos,title,thr=0.05):
    if not HAS_NX: ax.text(0.5,0.5,"networkx absent",ha="center");ax.set_title(title);return
    A=mask_diag(A);q=A.shape[0];Gb=np.abs(G)>0;wmax=np.abs(A).max() or 1.0
    Dg=nx.DiGraph(); [Dg.add_node(i) for i in range(q)]
    for i in range(q):
        for j in range(q):
            if abs(A[i,j])>thr: Dg.add_edge(i,j,w=abs(A[i,j]),phys=bool(Gb[i,j]))
    P={i:pos.get(i,(np.cos(i),np.sin(i))) for i in range(q)}
    nx.draw_networkx_nodes(Dg,P,ax=ax,node_color="#dfe6e9",edgecolors="#2d3436",node_size=900)
    nx.draw_networkx_labels(Dg,P,{i:labels[i] for i in range(q)},ax=ax,font_size=7)
    for (i,j,d) in Dg.edges(data=True):
        ax.annotate("",xy=P[j],xytext=P[i],arrowprops=dict(arrowstyle="-|>",lw=1+3*d["w"]/wmax,
            color=("#2ecc71" if d["phys"] else "#e67e22"),linestyle=("solid" if d["phys"] else "dashed"),
            alpha=0.85,shrinkA=16,shrinkB=16))
    ax.set_title(title,fontsize=11);ax.axis("off")
if A is not None:
    fig,ax=plt.subplots(figsize=(7.5,6.5))
    draw_dag(ax,A,G,LABELS,POS,f"{NAME}")
    import matplotlib.patches as mp
    fig.legend(handles=[mp.Patch(color="#2ecc71",label="arête physique"),mp.Patch(color="#e67e22",label="arête extra")],
               loc="lower center",ncol=2,bbox_to_anchor=(0.5,-0.02))
    fig.suptitle("G3 — DAG appris (réseau) — largeur ∝ |poids|",y=1.01,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/"G3_network.png",bbox_inches="tight");plt.show()
else: print("[G3 saute]")


In [ ]:
# === Niveau A — G4 : force des arêtes (physique vs extra) ===
if A is not None:
    Ad=mask_diag(A);q=Ad.shape[0];Gb=np.abs(G)>0;thr=0.05;rows=[]
    for i in range(q):
        for j in range(q):
            if abs(Ad[i,j])>thr: rows.append((f"{LABELS[i]}→{LABELS[j]}",abs(Ad[i,j]),bool(Gb[i,j])))
    rows.sort(key=lambda r:-r[1])
    fig,ax=plt.subplots(figsize=(7,max(3,0.32*len(rows))))
    ax.barh(range(len(rows)),[r[1] for r in rows],color=["#2ecc71" if r[2] else "#e67e22" for r in rows])
    ax.set_yticks(range(len(rows)));ax.set_yticklabels([r[0] for r in rows],fontsize=7);ax.invert_yaxis()
    ax.set_xlabel("|poids A_dag|");ax.set_title(f"G4 — Force des arêtes — {NAME} (vert=physique, orange=extra)",fontsize=11)
    plt.tight_layout();plt.savefig(FIG_DIR/"G4_edge_bars.png",bbox_inches="tight");plt.show()
else: print("[G4 saute]")


In [ ]:
# === Niveau A — G5 : distribution des poids (signature verrouillage prior) ===
if A is not None:
    w=np.abs(mask_diag(A)); w=w[w>1e-4]
    fig,ax=plt.subplots(figsize=(6,3.8))
    ax.hist(w,bins=30,color="#0984e3",alpha=0.85)
    ax.set_title(f"G5 — {NAME} — |poids A_dag| non nuls\n(pic serré = squelette figé / prior-lock)",fontsize=10)
    ax.set_xlabel("|poids|");ax.set_ylabel("compte")
    plt.tight_layout();plt.savefig(FIG_DIR/"G5_weight_hist.png",bbox_inches="tight");plt.show()
else: print("[G5 saute]")


In [ ]:
# === Niveau A — G6 : degrés + parents de SP_HR (drivers de la précip) ===
if A is not None:
    Ad=mask_diag(A);Ab=(np.abs(Ad)>0.05).astype(int)
    fig,ax=plt.subplots(1,2,figsize=(13,4.2))
    x=np.arange(len(LABELS));w=0.4
    ax[0].bar(x-w/2,Ab.sum(1),w,label="sortant",color="#6c5ce7")
    ax[0].bar(x+w/2,Ab.sum(0),w,label="entrant",color="#00b894")
    ax[0].set_xticks(x);ax[0].set_xticklabels(LABELS,rotation=90,fontsize=7);ax[0].legend(fontsize=8)
    ax[0].set_title(f"Degrés — {NAME}",fontsize=10)
    if "SP_HR" in LABELS:
        j=LABELS.index("SP_HR");col=np.abs(Ad[:,j]);idx=[i for i in np.argsort(-col) if col[i]>0.05]
        ax[1].barh([LABELS[i] for i in idx],[col[i] for i in idx],color="#0984e3");ax[1].invert_yaxis()
        ax[1].set_xlabel("|poids| vers SP_HR");ax[1].set_title(f"Parents de SP_HR (précip) — {NAME}",fontsize=10)
    fig.suptitle("G6 — Topologie : hubs & drivers de la précipitation",y=1.02,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/"G6_degrees.png",bbox_inches="tight");plt.show()
else: print("[G6 saute]")


In [ ]:
# === Niveau A — G7 : métriques d'explicabilité (table + barres) ===
if M is not None:
    keys=["q_phys_binary","q_phys_cont","skeleton_f1","n_extra","sign_correct"]
    vals=[M.get(k,0) for k in keys]
    fig,ax=plt.subplots(figsize=(8,3.6))
    bars=ax.bar(keys,[v if isinstance(v,(int,float)) else 0 for v in vals],color="#e17055")
    for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2,b.get_height(),f"{v:.2f}" if isinstance(v,float) else str(v),ha="center",va="bottom",fontsize=9)
    ax.set_title(f"G7 — Métriques d'explicabilité — {NAME} (↑ mieux sauf n_extra ↓)",fontsize=11)
    plt.xticks(rotation=20,fontsize=8)
    plt.tight_layout();plt.savefig(FIG_DIR/"G7_metrics.png",bbox_inches="tight");plt.show()
    print("Table:", {k:M.get(k) for k in keys})
else: print("[G7 saute]")


## Niveau B — Prédictions (depuis les caches, sans sampling)

Depuis le cache Stage-1 (`mu_HR`, `baseline_log`, `delta_target`, `valid_mask`) :
- **cible** = `baseline_log + mu_HR + delta_target`
- **prédiction moyenne** = `baseline_log + mu_HR` (le résidu de diffusion est de moyenne ~nulle)

Rapide, pas de GPU. *(Nécessite le cache — Colab/Drive.)*


In [ ]:
# === Niveau B — chargement du cache ===
def load_cache(p):
    if not (HAS_TORCH and Path(p).exists()): print("[cache absent]",p); return None
    c=_tload(p); print(f"[cache] {Path(p).name} keys=",list(c.keys())); return c
CC = load_cache(CACHE)

def fields(c):
    if c is None: return None
    def g(k):
        v=c.get(k)
        if v is None: return None
        if hasattr(v,"numpy"): return v[:MAX_SAMPLES].float().numpy()
        return np.asarray(v)[:MAX_SAMPLES]
    mu=g("mu_HR");bl=g("baseline_log");dt=g("delta_target");vm=g("valid_mask")
    if mu is None or bl is None or dt is None: print("  [warn] clefs manquantes"); return None
    tgt=bl+mu+dt; mean=bl+mu
    if vm is not None:
        m=vm>0.5; tgt=np.where(m,tgt,np.nan); mean=np.where(m,mean,np.nan)
    return dict(target=tgt,mean=mean,mu=mu,baseline=bl,delta=dt)
F = fields(CC)
print("F:", None if F is None else F["target"].shape)


In [ ]:
# === Niveau B — G8 : cartes cible / prédiction moyenne / erreur ===
if F is not None:
    N=F["target"].shape[0]; idx=np.linspace(0,N-1,min(3,N)).astype(int)
    fig,axes=plt.subplots(len(idx),3,figsize=(11,3.4*len(idx)));axes=np.atleast_2d(axes)
    for r,s in enumerate(idx):
        t=F["target"][s,0];m=F["mean"][s,0];e=m-t
        for c,(img,ttl,cm) in enumerate([(t,"cible","viridis"),(m,"préd. moyenne","viridis"),(e,"erreur","RdBu_r")]):
            v=np.nanmax(np.abs(img)) or 1
            im=axes[r,c].imshow(img,cmap=cm,vmin=(-v if c==2 else None),vmax=(v if c==2 else None),origin="lower")
            axes[r,c].set_title(f"{ttl} #{s}",fontsize=9);axes[r,c].axis("off");plt.colorbar(im,ax=axes[r,c],fraction=0.046)
    fig.suptitle(f"G8 — {NAME} — cible / prédiction / erreur (log)",y=1.01,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/f"G8_maps_{slug(NAME)}.png",bbox_inches="tight");plt.show()
else: print("[G8 saute] cache absent")


In [ ]:
# === Niveau B — G9 : scatter préd vs cible + queue des extrêmes ===
if F is not None:
    t=F["target"].ravel();m=F["mean"].ravel();ok=~(np.isnan(t)|np.isnan(m));t,m=t[ok],m[ok]
    if t.size>200000:
        s=np.random.default_rng(0).choice(t.size,200000,replace=False);t,m=t[s],m[s]
    fig,ax=plt.subplots(1,2,figsize=(11,4))
    ax[0].hexbin(t,m,gridsize=60,cmap="viridis",bins="log");lim=[min(t.min(),m.min()),max(t.max(),m.max())]
    ax[0].plot(lim,lim,"r--",lw=1);ax[0].set_xlabel("cible");ax[0].set_ylabel("préd moyenne")
    ax[0].set_title(f"{NAME} — préd vs cible (r={np.corrcoef(t,m)[0,1]:.3f})",fontsize=10)
    p=[90,95,99,99.9]
    ax[1].plot(p,[np.percentile(t,q) for q in p],"o-",label="cible")
    ax[1].plot(p,[np.percentile(m,q) for q in p],"s--",label="préd moyenne")
    ax[1].set_xlabel("percentile");ax[1].set_ylabel("valeur (log)");ax[1].legend();ax[1].set_title(f"{NAME} — queue",fontsize=10)
    plt.tight_layout();plt.savefig(FIG_DIR/f"G9_scatter_{slug(NAME)}.png",bbox_inches="tight");plt.show()
else: print("[G9 saute]")


In [ ]:
# === Niveau B — G10 : biais moyen + RMSE par pixel ===
if F is not None:
    err=F["mean"]-F["target"]
    bias=np.nanmean(err,axis=0)[0];rmse=np.sqrt(np.nanmean(err**2,axis=0))[0]
    fig,ax=plt.subplots(1,2,figsize=(10,4));v=np.nanmax(np.abs(bias)) or 1
    im0=ax[0].imshow(bias,cmap="RdBu_r",vmin=-v,vmax=v,origin="lower");ax[0].set_title(f"{NAME} — biais moyen",fontsize=10);plt.colorbar(im0,ax=ax[0],fraction=0.046)
    im1=ax[1].imshow(rmse,cmap="magma",origin="lower");ax[1].set_title(f"{NAME} — RMSE/pixel",fontsize=10);plt.colorbar(im1,ax=ax[1],fraction=0.046)
    for a in ax: a.axis("off")
    plt.tight_layout();plt.savefig(FIG_DIR/f"G10_err_{slug(NAME)}.png",bbox_inches="tight");plt.show()
else: print("[G10 saute]")


## Niveau C — (optionnel) Sampling + ablation A_dag

Recharge le stack et mesure l'**ablation A_dag** (A_dag=0 → Δ/signal) : le vrai test « le DAG conditionne-t-il la sortie ? ».

⚠️ Colab + checkpoint requis. `RUN_TIER_C = True` pour activer. Reconstruire le stack via le factory du notebook d'entraînement (ne pas réimplémenter).


In [ ]:
# === Niveau C — scaffold (désactivé) ===
RUN_TIER_C = False
if RUN_TIER_C:
    def ablation_delta(stack, batch, predict_fn, K=1, n_steps=18):
        rcn=stack["rcn_cell"]; A0=rcn.A_dag.detach().clone()
        full=predict_fn(stack,batch,K=K,n_steps=n_steps).nanmean(0).squeeze().cpu().numpy()
        rcn.A_dag.data.zero_()
        abl =predict_fn(stack,batch,K=K,n_steps=n_steps).nanmean(0).squeeze().cpu().numpy()
        rcn.A_dag.data.copy_(A0)
        delta=abl-full; ratio=float(np.abs(delta).mean()/max(np.abs(full).mean(),1e-12))
        fig,ax=plt.subplots(1,3,figsize=(12,4))
        for a,(img,t,cm) in zip(ax,[(full,"A_dag appris","viridis"),(abl,"A_dag=0","viridis"),(delta,f"Δ (Δ/signal={ratio:.1%})","RdBu_r")]):
            v=np.nanmax(np.abs(img)) or 1;im=a.imshow(img,cmap=cm,vmin=(-v if cm=="RdBu_r" else None),vmax=(v if cm=="RdBu_r" else None),origin="lower")
            a.set_title(t,fontsize=10);a.axis("off");plt.colorbar(im,ax=a,fraction=0.046)
        plt.tight_layout();plt.savefig(FIG_DIR/"GC_ablation.png",bbox_inches="tight");plt.show()
        return ratio
    print("Niveau C prêt : ablation_delta(stack, batch, predict_with_stack).")
else:
    print("Niveau C désactivé (RUN_TIER_C=False).")


In [ ]:
# === Dossier copiable : zip final ===
import shutil
zip_path = shutil.make_archive(str(FIG_DIR), "zip", root_dir=str(FIG_DIR))
figs = sorted(p.name for p in Path(FIG_DIR).glob("*.png"))
print(f"{len(figs)} figures dans {FIG_DIR}")
for f in figs: print("  -", f)
print("\nZIP copiable ->", zip_path)
try:
    if IN_COLAB:
        from google.colab import files; files.download(zip_path)
except Exception as e:
    print("(téléchargement auto indispo:",e,") — copie le .zip depuis Drive.")


## Lecture des résultats
- **G1–G7 (structurel)** : `skeleton_f1≈1`, `n_extra≈0`, arêtes vertes → DAG fidèle à la physique. Poids ~uniformes (G5) → **verrouillage prior** (imposé, pas découvert).
- **G8–G10 (prédictions)** : champ moyen, biais, queue. Le résidu stochastique n'y est pas — voir Niveau C.
- **Niveau C** : `Δ/signal` faible → le décodeur **ignore largement A_dag** (conditionnement décoratif).

> Structure **fidèle et auditable** ≠ mécanisme causal **à fort levier**. Revendiquer « physique imposée », pas « découverte ».
